In [62]:
# imports
import os, random, pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)

TensorFlow: 2.20.0


In [63]:
# load broken dataset
with open('mnist.hupsista','rb') as f:
    X_test_broken, y_test_broken = pickle.load(f)

X_test_broken = X_test_broken.astype('float32')
# normalize
if X_test_broken.max() > 1.0:
    X_test_broken /= 255.0
# channel dimension
if X_test_broken.ndim == 3:
    X_test_broken = X_test_broken[..., None]

unique_labels = np.unique(y_test_broken)
print(f'Broken test data shape: {X_test_broken.shape}')
print(f'Unique labels in broken test set: {unique_labels}')

Broken test data shape: (10000, 28, 28, 1)
Unique labels in broken test set: [0. 1. 2. 3. 4. 5. 6. 7. 8. 9.]


In [64]:
# load mnist training data
print("\n--- Loading clean MNIST training data ---")
(x_train_clean, y_train_clean), (_, _) = tf.keras.datasets.mnist.load_data()

# use inly half of dataset
x_train_clean = x_train_clean[:30000]
y_train_clean = y_train_clean[:30000]

# reprocess training data
x_train_clean = x_train_clean.astype('float32') / 255.0  # normalize
x_train_clean = x_train_clean[..., np.newaxis]  # channel dimension
y_train_clean = y_train_clean.astype(np.int32)

# shape
print(f'Clean training data shape: {x_train_clean.shape}')
print(f'Clean training labels shape: {y_train_clean.shape}')


--- Loading clean MNIST training data ---
Clean training data shape: (30000, 28, 28, 1)
Clean training labels shape: (30000,)


In [65]:
# split to training/validation
X_train_clean, X_val_clean, y_train_clean, y_val_clean = train_test_split(
    x_train_clean, y_train_clean, test_size=0.1, random_state=SEED, stratify=y_train_clean
)

print(f'Train (clean): {X_train_clean.shape}')
print(f'Validation (clean): {X_val_clean.shape}')
print(f'Test (broken): {X_test_broken.shape}')

Train (clean): (27000, 28, 28, 1)
Validation (clean): (3000, 28, 28, 1)
Test (broken): (10000, 28, 28, 1)


In [66]:

def create_model(optimizer):
    # create model
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(28,28,1)),

        layers.RandomRotation(0.2),
        layers.RandomTranslation(0.1,0.1),
        layers.GaussianNoise(0.2),

        layers.Conv2D(16,3,activation="relu",padding="same"),
        layers.MaxPooling2D(),

        layers.Conv2D(32,3,activation="relu",padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64,3,activation="relu",padding="same"),
        layers.BatchNormalization(),

        layers.GlobalAveragePooling2D(),

        layers.Dense(10,activation="softmax")
    ])

    # compile model
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# early stopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# learning rate sheduler
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-6,        
    verbose=1
)



# display model architecture
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_rotation_2               │ (None, 28, 28, 1)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation_2            │ (None, 28, 28, 1)      │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gaussian_noise_2                │ (None, 28, 28, 1)      │             0 │
│ (GaussianNoise)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 14, 14, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 14, 14, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 7, 7, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 7, 7, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 7, 7, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48,470 (189.34 KB)

 Trainable params: 24,138 (94.29 KB)

 Non-trainable params: 192 (768.00 B)

 Optimizer params: 24,140 (94.30 KB)

In [67]:
# trains model on clean data
print("\n--- Training on clean MNIST data ---")
history = model.fit(
    X_train_clean, y_train_clean, 
    epochs=100, 
    batch_size=256,
    validation_data=(X_val_clean, y_val_clean), 
    verbose=1,
    callbacks=[early_stop, reduce_lr]
)


--- Training on clean MNIST data ---
Epoch 1/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.9639 - loss: 0.1249 - val_accuracy: 0.9657 - val_loss: 0.1089 - learning_rate: 1.0000e-06
Epoch 2/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.9631 - loss: 0.1228 - val_accuracy: 0.9660 - val_loss: 0.1090 - learning_rate: 1.0000e-06
Epoch 3/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.9624 - loss: 0.1235 - val_accuracy: 0.9657 - val_loss: 0.1088 - learning_rate: 1.0000e-06
Epoch 4/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - accuracy: 0.9639 - loss: 0.1229 - val_accuracy: 0.9657 - val_loss: 0.1091 - learning_rate: 1.0000e-06
Epoch 5/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.9640 - loss: 0.1242 - val_accuracy: 0.9660 - val_loss: 0.1092 - learning_rate: 1.0000e-06
Epoch 6/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - accuracy: 0.9633 - loss: 0.1247 - val_accuracy: 0.9660 - val_loss: 0.1090 - learning_rate: 1.0000e-06
Epoch 7/100
10

In [68]:
# save model to keras format
model.save("my_model.keras")
print("\nModel saved as 'my_model.keras'")


Model saved as 'my_model.keras'


In [69]:
# test model on broken dataset 
print("\n--- Evaluating on broken test data ---")
y_pred_probs = model.predict(X_test_broken, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

correct = np.sum(y_pred == y_test_broken)
total_test = len(y_test_broken)
accuracy = correct / total_test * 100

# calculate reward
if correct <= 5000:
    reward = correct * 100
elif correct <= 6000:
    reward = 5000 * 100 + (correct - 5000) * 200
else:
    reward = 5000 * 100 + 1000 * 200 + (correct - 6000) * 1000

# model parameters
params = model.count_params()

# calculate net score
net_score = reward - params

# results
print(f"\n{'='*50}")
print(f"RESULTS")
print(f"{'='*50}")
print(f"Correct predictions on broken test set: {correct}/{total_test} ({accuracy:.2f}%)")
print(f"Reward: {reward} €")
print(f"Model parameters: {params}")
print(f"Net Score: {net_score} €")
print(f"{'='*50}")


--- Evaluating on broken test data ---

RESULTS
Correct predictions on broken test set: 5098/10000 (50.98%)
Reward: 519600 €
Model parameters: 24330
Net Score: 495270 €


In [70]:
optimizers = [
    tf.keras.optimizers.SGD,
    tf.keras.optimizers.Adam,
    tf.keras.optimizers.RMSprop
]

learning_rates = [0.1, 0.01, 0.001]

def optimizer_vs_accuracy(model_fn, optimizers, learning_rates, x_train, y_train, x_test, y_test, epochs=5):

    results = {}

    for opt_class in optimizers:
        for lr in learning_rates:

            optimizer = opt_class(learning_rate=lr)

            model = model_fn(optimizer)

            model.fit(x_train, y_train, epochs=epochs, verbose=0)

            loss, acc = model.evaluate(x_test, y_test, verbose=0)

            name = f"{opt_class.__name__}_lr{lr}"

            print(f"{name} -> Accuracy: {acc:.4f}")

            results[name] = acc

    best = max(results, key=results.get)

    print("\nBest configuration:")
    print(best, "Accuracy:", results[best])

    return results

results = optimizer_vs_accuracy(
    create_model,
    optimizers,
    learning_rates,
    X_train_clean,
    y_train_clean,
    X_test_broken,
    y_test_broken,
    epochs=5
)

SGD_lr0.1 -> Accuracy: 0.4549
SGD_lr0.01 -> Accuracy: 0.4371
SGD_lr0.001 -> Accuracy: 0.1508
Adam_lr0.1 -> Accuracy: 0.4689
Adam_lr0.01 -> Accuracy: 0.4895
Adam_lr0.001 -> Accuracy: 0.4906
RMSprop_lr0.1 -> Accuracy: 0.3784
RMSprop_lr0.01 -> Accuracy: 0.3962
RMSprop_lr0.001 -> Accuracy: 0.5023

Best configuration:
RMSprop_lr0.001 Accuracy: 0.5023000240325928
